# SEEA Keyword Configuration and Validation
## a. Create `seea_keyword_config.json`

### Purpose

The file `seea_keyword_config.json` was created as a conservative keyword configuration for identifying possible evidence of SEEA account availability in official documents. We use it as a dictionary-based text-mining input, not as a certified translation or legal reference.

The configuration is designed to reduce false positives by requiring keyword evidence to combine two types of terms:

1. An account or accounting term, such as `account`, `accounts`, or translated equivalents.
2. A SEEA account topic term, such as `water emission`, `physical energy supply and use`, `ecosystem extent`, or translated equivalents.

Topic-only words, such as water, energy, forest, land, carbon, ocean, species, taxes, or subsidies, should not be treated as sufficient evidence of availability when they appear alone.

### File structure

The JSON file contains four main components.

| No | Field | Description |
|---:|---|---|
| 1 | `metadata` | Documents the purpose, research-use note, matching principle, country-language scope, translation scope, justification, and recommended validation steps. |
| 2 | `country_languages` | Maps each country to English plus the main official or commonly used language or languages for statistical publications. |
| 3 | `account_words_by_language` | Lists account, accounts, accounting, and equivalent terms by language. These words are used to ensure that a topic match is connected to accounting language. |
| 4 | `topic_translations` | Lists SEEA account topics and their keyword translations by language. These are used to generate account-topic keyword combinations. |

### Creation logic

The configuration was created using the following logic:

1. Start from the countries included in the 2024 Global Assessment sheet.
2. Assign each country to English and the main official or commonly used statistical-publication language or languages.
3. Add account or accounting words for every language listed in `country_languages`.
4. Add SEEA account-topic keywords under `topic_translations`.
5. Use complete topic coverage for English, Spanish, French, and Portuguese.
6. Add broader coverage for German and several European languages where terms are more reliable.
7. Use partial coverage for languages where unvalidated literal translation could reduce research reliability.
8. Preserve all generated phrases in the downstream keyword dictionary output so the screening process remains transparent and reproducible.

### Research-use interpretation

We should describe this file as a transparent and reproducible screening instrument. We should not describe it as definitive proof that a country has or does not have a SEEA account. Final availability claims should still be reviewed manually, especially when `nonreport_avail = 1` or when evidence is based on non-English keyword matches.

## b. Validate the keyword configuration

### Validation script

The keyword configuration was validated using:

```text
A_validate_seea_keyword_config.py
```

The script reads:

```text
seea_keyword_config.json
```

and writes:

```text
seea_keyword_config_coverage.csv
```

### Validation checks

The script checks the internal coverage of the JSON file by calculating language-level coverage. Specifically, it:

1. Loads `country_languages`, `account_words_by_language`, and `topic_translations` from the JSON file.
2. Identifies all countries included in the configuration.
3. Identifies all languages used across countries.
4. Identifies all SEEA account types included in `topic_translations`.
5. Counts which countries use each language.
6. Checks whether each language has account words.
7. Counts how many SEEA account types have topic translations for each language.
8. Calculates the percentage of topic-translation coverage by language.
9. Exports the results to `seea_keyword_config_coverage.csv`.

### Output file

The validation output includes the following fields.

| No | Field | Description |
|---:|---|---|
| 1 | `language` | Language identified from the country-language mapping. |
| 2 | `countries_using_language` | Countries in the configuration that use the language. |
| 3 | `country_count` | Number of countries using the language. |
| 4 | `has_account_words` | Indicator for whether the language has account or accounting terms. |
| 5 | `account_word_count` | Number of account or accounting words available for the language. |
| 6 | `topic_translation_account_types` | Number of SEEA account types with topic translations in that language. |
| 7 | `topic_translation_total_account_types` | Total number of SEEA account types included in the configuration. |
| 8 | `topic_translation_coverage_pct` | Percentage of SEEA account types covered by topic translations for the language. |

### Console summary

The validation script also prints a short console summary:

```text
Countries in config: [number]
Languages in config: [number]
Account types with topic translations: [number]
Languages missing account words: [list]
Countries not in config: [list]
Wrote seea_keyword_config_coverage.csv
```

For the current configuration, the file includes:

| No | Description | Value |
|---:|---|---:|
| 1 | Countries in `country_languages` | 120 |
| 2 | Languages with account words | 52 |
| 3 | SEEA account types in `topic_translations` | 36 |

### How to run the validation

Place the following files in the same working directory:

```text
seea_keyword_config.json
A_validate_seea_keyword_config.py
```

Then run:

```bash
python A_validate_seea_keyword_config.py
```

After the script finishes, review:

```text
seea_keyword_config_coverage.csv
```



In [1]:
import json
import pandas as pd


CONFIG_JSON = "seea_keyword_config.json"
OUTPUT_CSV = "seea_keyword_config_coverage.csv"


def main():
    with open(CONFIG_JSON, "r", encoding="utf-8") as file:
        config = json.load(file)

    country_languages = config["country_languages"]
    account_words = config["account_words_by_language"]
    topic_translations = config["topic_translations"]

    countries = sorted(country_languages)
    languages = sorted({language for values in country_languages.values() for language in values})
    account_types = sorted(topic_translations)

    rows = []
    for language in languages:
        countries_using_language = [country for country, values in country_languages.items() if language in values]
        topic_count = sum(1 for account_type in account_types if language in topic_translations.get(account_type, {}))
        rows.append(
            {
                "language": language,
                "countries_using_language": "; ".join(countries_using_language),
                "country_count": len(countries_using_language),
                "has_account_words": int(language in account_words),
                "account_word_count": len(account_words.get(language, [])),
                "topic_translation_account_types": topic_count,
                "topic_translation_total_account_types": len(account_types),
                "topic_translation_coverage_pct": round(topic_count / len(account_types) * 100, 1),
            }
        )

    missing_account_word_languages = [language for language in languages if language not in account_words]
    countries_not_in_config = []

    pd.DataFrame(rows).to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

    print(f"Countries in config: {len(countries)}")
    print(f"Languages in config: {len(languages)}")
    print(f"Account types with topic translations: {len(account_types)}")
    print(f"Languages missing account words: {missing_account_word_languages}")
    print(f"Countries not in config: {countries_not_in_config}")
    print(f"Wrote {OUTPUT_CSV}")


if __name__ == "__main__":
    main()


Countries in config: 120
Languages in config: 52
Account types with topic translations: 36
Languages missing account words: []
Countries not in config: []
Wrote seea_keyword_config_coverage.csv


### Manual review after validation

The coverage file should be used to identify languages with lower topic-translation coverage and to prioritize manual review. We should manually inspect the keyword dictionary and availability evidence before using the results as final analytical claims. This is especially important for high-impact findings, non-English matches, and cases where availability is inferred from downloaded files rather than official reporting tables.
